# 論文用 3 特徴量比較図生成

この notebook は、`results/comparison/<dataset>/comparison_summary.csv` から `overall` と prefix 群の `packet_count_median` / `byte_count_median` / `duration_median` を横並びで比較する図だけを生成します。

元 pcap や flow CSV の再処理は行いません。出力先は `results/paper_figures/<dataset>/` です。


In [ ]:
# 設定セル: 図生成前にここだけ変更してください。
DATASET = "202604081400"  # None の場合は results/ から自動選択。
OUTPUT_DIR = None          # None なら results/paper_figures/<dataset>/
FORMATS = ["pdf", "png"]
OVERWRITE = False

MIN_PREFIX_FLOW_COUNT = 1000
MAX_PREFIXES_TO_PLOT = 10
SORT_BY = "flow_count"  # "flow_count" or "byte_count_median"
PLOT_LOG_SCALE_VERSION = True


In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import matplotlib

try:
    get_ipython
except NameError:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.size": 10,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
})


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("project root not found: expected AGENTS.md and results/")


PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "results"
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


def rel(path: Path | None) -> str:
    if path is None:
        return ""
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def available_datasets() -> pd.DataFrame:
    comparison_root = RESULTS_DIR / "comparison"
    rows = []
    for path in sorted(comparison_root.glob("*/comparison_summary.csv")) if comparison_root.exists() else []:
        rows.append({"dataset": path.parent.name, "comparison_summary": True})
    return pd.DataFrame(rows)


def choose_dataset(configured: str | None) -> str:
    datasets = available_datasets()
    if configured:
        if datasets.empty or configured not in set(datasets["dataset"]):
            print(f"[warn] configured dataset not discovered under results/comparison: {configured}")
        return configured
    if datasets.empty:
        raise FileNotFoundError("no comparison_summary.csv found under results/comparison/")
    return str(datasets.sort_values("dataset").iloc[-1]["dataset"])


def target_col(df: pd.DataFrame) -> str | None:
    for candidate in ["target", "normalized_dst_prefix", "dst_prefix", "prefix", "aggregate_id"]:
        if candidate in df.columns:
            return candidate
    lowered = {column.lower(): column for column in df.columns}
    for candidate in ["target", "normalized_dst_prefix", "dst_prefix", "prefix", "aggregate_id"]:
        if candidate in lowered:
            return lowered[candidate]
    return None


def ensure_output_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def output_path(outdir: Path, stem: str, ext: str, overwrite: bool) -> Path:
    ext = ext.lstrip(".")
    base = outdir / f"{stem}.{ext}"
    if overwrite or not base.exists():
        return base
    for i in range(1, 1000):
        candidate = outdir / f"{stem}_{i:02d}.{ext}"
        if not candidate.exists():
            return candidate
    raise FileExistsError(f"too many existing files for {base}")


def polish_figure_for_export(fig: plt.Figure, *, left: float = 0.09, bottom: float = 0.34, w_pad: float = 2.4) -> None:
    try:
        fig.align_ylabels(fig.axes)
    except Exception:
        pass
    fig.tight_layout(pad=1.1, w_pad=w_pad, h_pad=1.0)
    fig.subplots_adjust(left=left, bottom=bottom)


def save_current(fig: plt.Figure, outdir: Path, stem: str, formats: list[str], overwrite: bool) -> list[Path]:
    ensure_output_dir(outdir)
    saved = []
    for fmt in formats:
        path = output_path(outdir, stem, fmt, overwrite)
        fig.savefig(path, bbox_inches="tight", pad_inches=0.12)
        saved.append(path)
    return saved


In [ ]:
def normalize_prefix_label(label: str) -> str:
    """Convert internal target names such as dst_202.244.127.0_24 to CIDR-like labels."""
    text = str(label).strip()
    if text.lower() == "overall":
        return "overall"
    text = re.sub(r"^(src|dst)_", "", text)
    if "/" in text:
        return text
    sanitized_ipv6_match = re.fullmatch(r"([0-9A-Fa-f_]+)___(\d{1,3})", text)
    if sanitized_ipv6_match:
        addr = sanitized_ipv6_match.group(1).replace("_", ":") + "::"
        return f"{addr}/{sanitized_ipv6_match.group(2)}"
    ipv4_match = re.fullmatch(r"((?:\d{1,3}\.){3}\d{1,3})_(\d{1,2})", text)
    if ipv4_match:
        return f"{ipv4_match.group(1)}/{ipv4_match.group(2)}"
    ipv6_match = re.fullmatch(r"([0-9A-Fa-f:]+)_(\d{1,3})", text)
    if ipv6_match and ":" in ipv6_match.group(1):
        return f"{ipv6_match.group(1)}/{ipv6_match.group(2)}"
    return text


def prefix_letter_label(index: int) -> str:
    alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    label = ""
    value = int(index)
    while True:
        value, remainder = divmod(value, len(alphabet))
        label = alphabet[remainder] + label
        if value == 0:
            return label
        value -= 1


def apply_anonymous_prefix_axis_labels(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    labels = []
    prefix_index = 0
    for _, row in out.iterrows():
        raw_label = str(row.get("prefix", row.get("display_label", ""))).strip()
        if raw_label.lower() == "overall":
            labels.append("overall")
        else:
            labels.append(prefix_letter_label(prefix_index))
            prefix_index += 1
    out["display_label"] = labels
    return out


def comparison_summary_path(dataset: str) -> Path:
    return RESULTS_DIR / "comparison" / dataset / "comparison_summary.csv"


def coerce_summary_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    rename_map = {
        "median_duration": "duration_median",
        "median_packet_count": "packet_count_median",
        "median_byte_count": "byte_count_median",
        "median_avg_packet_size": "avg_packet_size_median",
    }
    for old, new in rename_map.items():
        if old in out.columns and new not in out.columns:
            out[new] = out[old]
    for col in [
        "flow_count",
        "duration_median",
        "packet_count_median",
        "byte_count_median",
        "avg_packet_size_median",
        "tcp_ratio",
        "udp_ratio",
        "tcp_flow_ratio",
        "udp_flow_ratio",
    ]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def load_overall_summary(dataset: str) -> pd.DataFrame:
    path = comparison_summary_path(dataset)
    if not path.exists():
        raise FileNotFoundError(f"comparison summary not found: {rel(path)}")
    df = coerce_summary_columns(pd.read_csv(path))
    tcol = target_col(df)
    if tcol is None:
        raise ValueError(f"target column not found in {rel(path)}")
    overall = df[df[tcol].astype(str).str.lower() == "overall"].copy()
    if overall.empty:
        raise ValueError(f"overall row not found in {rel(path)}")
    overall["prefix"] = "overall"
    overall["display_label"] = "overall"
    return overall.reset_index(drop=True)


def load_prefix_summaries(dataset: str, min_flow_count: int = 1000) -> pd.DataFrame:
    path = comparison_summary_path(dataset)
    if not path.exists():
        raise FileNotFoundError(f"comparison summary not found: {rel(path)}")
    df = coerce_summary_columns(pd.read_csv(path))
    tcol = target_col(df)
    if tcol is None:
        raise ValueError(f"target column not found in {rel(path)}")
    prefix_df = df[df[tcol].astype(str).str.lower() != "overall"].copy()
    prefix_df = prefix_df.rename(columns={tcol: "prefix"})
    if "flow_count" in prefix_df.columns:
        prefix_df = prefix_df[prefix_df["flow_count"].fillna(0) >= min_flow_count]
    else:
        print(f"[warn] flow_count column not found in {rel(path)}; no flow-count filter applied")
    prefix_df["display_label"] = prefix_df["prefix"].map(normalize_prefix_label)
    return prefix_df.reset_index(drop=True)


def required_median_columns(df: pd.DataFrame) -> list[str]:
    metrics = ["packet_count_median", "byte_count_median", "duration_median"]
    available = [metric for metric in metrics if metric in df.columns and pd.to_numeric(df[metric], errors="coerce").notna().any()]
    missing = [metric for metric in metrics if metric not in available]
    if missing:
        raise ValueError(f"required median columns missing or empty: {missing}")
    return available


In [ ]:
def plot_prefix_packet_byte_duration_median_comparison(
    overall_df: pd.DataFrame,
    prefix_df: pd.DataFrame,
    output_dir: Path,
    max_prefixes: int = 8,
    sort_by: str = "flow_count",
    use_log_y: bool = False,
) -> pd.DataFrame:
    """Plot packet, byte, and duration medians side by side."""
    if overall_df.empty or prefix_df.empty:
        raise ValueError("overall or prefix summary is empty")

    metrics = required_median_columns(pd.concat([overall_df, prefix_df], ignore_index=True, sort=False))
    work_prefix = prefix_df.copy()
    if sort_by not in work_prefix.columns:
        print(f"[warn] SORT_BY={sort_by!r} not found; falling back to flow_count")
        sort_by = "flow_count" if "flow_count" in work_prefix.columns else metrics[0]
    work_prefix[sort_by] = pd.to_numeric(work_prefix[sort_by], errors="coerce")
    work_prefix = work_prefix.sort_values(sort_by, ascending=False).head(max_prefixes)

    overall_row = overall_df.iloc[[0]].copy()
    plot_df = pd.concat([overall_row, work_prefix], ignore_index=True, sort=False)
    plot_df["display_label"] = plot_df["display_label"].fillna(plot_df["prefix"].astype(str).map(normalize_prefix_label))
    plot_df = apply_anonymous_prefix_axis_labels(plot_df)

    fig, axes = plt.subplots(1, len(metrics), figsize=(11.4, 3.9), sharex=False)
    axes = np.atleast_1d(axes)
    colors = ["#5B5B5B"] + ["#4C78A8"] * (len(plot_df) - 1)
    x = np.arange(len(plot_df))

    ylabels = {
        "packet_count_median": "Median packets per flow",
        "byte_count_median": "Median bytes per flow",
        "duration_median": "Median flow duration [s]",
    }
    titles = {
        "packet_count_median": "Packet count",
        "byte_count_median": "Byte count",
        "duration_median": "Duration",
    }
    for ax, metric in zip(axes, metrics):
        values = pd.to_numeric(plot_df[metric], errors="coerce")
        ax.bar(x, values, color=colors, edgecolor="white", linewidth=0.6)
        ax.set_title(titles.get(metric, metric))
        ax.set_ylabel(ylabels.get(metric, metric))
        ax.set_xticks(x)
        ax.set_xticklabels(plot_df["display_label"], rotation=35, ha="right")
        ax.set_xlabel("prefix", labelpad=-14)
        if use_log_y and (values > 0).any():
            ax.set_yscale("log")
            ax.set_ylabel(f"{ylabels.get(metric, metric)} (log scale)")
        ax.margins(x=0.02)

    polish_figure_for_export(fig)
    suffix = "_logy" if use_log_y else ""
    saved_paths = save_current(
        fig,
        output_dir,
        f"prefix_median_packet_byte_duration_comparison{suffix}",
        FORMATS,
        OVERWRITE,
    )
    for saved_path in saved_paths:
        print("saved:", rel(saved_path))
    if plt.get_backend().lower() == "agg":
        plt.close(fig)
    else:
        plt.show()
    return plot_df


In [ ]:
dataset = choose_dataset(DATASET)
outdir = ensure_output_dir(Path(OUTPUT_DIR) if OUTPUT_DIR else RESULTS_DIR / "paper_figures" / dataset)

overall_summary = load_overall_summary(dataset)
prefix_summaries = load_prefix_summaries(dataset, min_flow_count=MIN_PREFIX_FLOW_COUNT)

print(f"dataset = {dataset}")
print(f"output directory = {rel(outdir)}")
print(f"prefix rows after flow_count >= {MIN_PREFIX_FLOW_COUNT}: {len(prefix_summaries)}")

display_columns = [
    "display_label",
    "prefix",
    "flow_count",
    "packet_count_median",
    "byte_count_median",
    "duration_median",
    "avg_packet_size_median",
    "tcp_ratio",
    "udp_ratio",
]
display(prefix_summaries[[c for c in display_columns if c in prefix_summaries.columns]].head(MAX_PREFIXES_TO_PLOT * 2))

plot_df = plot_prefix_packet_byte_duration_median_comparison(
    overall_summary,
    prefix_summaries,
    outdir,
    max_prefixes=MAX_PREFIXES_TO_PLOT,
    sort_by=SORT_BY,
    use_log_y=False,
)

if PLOT_LOG_SCALE_VERSION:
    plot_prefix_packet_byte_duration_median_comparison(
        overall_summary,
        prefix_summaries,
        outdir,
        max_prefixes=MAX_PREFIXES_TO_PLOT,
        sort_by=SORT_BY,
        use_log_y=True,
    )

display(plot_df[[c for c in display_columns if c in plot_df.columns]])

created = sorted([p for p in outdir.glob("prefix_median_packet_byte_duration_comparison*") if p.suffix.lower().lstrip(".") in set(FORMATS)])
print(f"created/available 3-feature figures: {len(created)}")
for path in created:
    print(rel(path))
